# News Headline Generator — LoRA Fine-Tuning (Task 2 / Bonus)

Fine-tunes `google/flan-t5-small` on the Gigaword headline-generation dataset
using LoRA (via PEFT). Designed to run on Colab's **free T4 GPU**.

**Before running:** Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU -> Save.

Estimated time on free T4: ~20-40 min for a small training subset (see `TRAIN_SUBSET_SIZE` below).

## 1. Install dependencies

In [ ]:
!pip install -q transformers datasets peft accelerate evaluate rouge_score sentencepiece
!pip uninstall -y torchao -q

## 2. Imports

In [ ]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
)
from peft import LoraConfig, get_peft_model, TaskType
import evaluate

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)
if DEVICE == "cpu":
    print("WARNING: no GPU detected. Go to Runtime -> Change runtime type -> T4 GPU.")

Using device: cuda


## 3. Load dataset

Using **Gigaword** — a standard headline-generation benchmark (article sentence -> headline).
We use a small subset so this fits comfortably in a free Colab session.

**If this cell still errors:** Hugging Face occasionally changes dataset paths. Search "gigaword huggingface parquet" to find the current working path, or swap in any other article->headline dataset with `document` and `summary` (or similarly named) fields.

In [ ]:
MODEL_NAME = "google/flan-t5-small"
TRAIN_SUBSET_SIZE = 3000
EVAL_SUBSET_SIZE = 300

raw_dataset = load_dataset("JulesBelveze/tldr_news")
print("Splits available:", list(raw_dataset.keys()))

sample = raw_dataset["train"][0]
print("Sample row:", sample)

# Auto-detect the (headline, article) column pair
possible_pairs = [("headline", "content"), ("title", "text"), ("document", "summary")]
cols = raw_dataset["train"].column_names
HEADLINE_COL, ARTICLE_COL = None, None
for h, c in possible_pairs:
    if h in cols and c in cols:
        HEADLINE_COL, ARTICLE_COL = h, c
        break

if ARTICLE_COL is None:
    raise ValueError(f"Could not auto-detect columns. Available columns: {cols}")
print(f"Using ARTICLE_COL='{ARTICLE_COL}', HEADLINE_COL='{HEADLINE_COL}'")

# Only a "train" split exists — carve out a validation slice from it
split = raw_dataset["train"].train_test_split(test_size=0.1, seed=42)
train_full = split["train"]
eval_full = split["test"]

TRAIN_SUBSET_SIZE = min(TRAIN_SUBSET_SIZE, len(train_full))
EVAL_SUBSET_SIZE = min(EVAL_SUBSET_SIZE, len(eval_full))

train_data = train_full.shuffle(seed=42).select(range(TRAIN_SUBSET_SIZE))
eval_data = eval_full.shuffle(seed=42).select(range(EVAL_SUBSET_SIZE))

print(f"Train size: {len(train_data)}, Eval size: {len(eval_data)}")
print(train_data[0])

Splits available: ['train']
Sample row: {'category': 'ai', 'section': 'Sponsor', 'title': 'Report: New data on navigating the AI adoption gap', 'text': "Around 2 out of 3 executives say generative AI adoption has created tension — 42% say it's tearing their company apart. For many enterprises, conflicts, silos, and even sabotage are stalling progress. Yet there's reason for optimism: 77% of employees using AI are, or could become, an AI champion. The Writer 2025 generative AI report explores the pain points and potential of AI in the enterprise. In partnership with Workplace Intelligence, we uncovered what leaders are struggling with, where there's opportunity, and what employees think about AI adoption efforts. Read the full report for more insights and takeaways.", 'url': 'https://srv.buysellads.com/ads/long/x/TCUHDYZFTTTTTTH4ZUWN5TTTTTTKBQGZKATTTTTTE4TO47YTTTTTTAD75TBDLEP2F3KI4BPZQ2PMWIZBVR7UCAQW2HHE', 'newsletter_url': 'https://tldr.tech/ai/2025-04-01'}
Using ARTICLE_COL='text', HE

## 4. Load tokenizer & base model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


## 5. Tokenize

In [ ]:
MAX_INPUT_LEN = 128
MAX_TARGET_LEN = 32
PREFIX = "generate headline: "

def preprocess(examples):
    inputs = [PREFIX + doc for doc in examples[ARTICLE_COL]]
    model_inputs = tokenizer(
        inputs, max_length=MAX_INPUT_LEN, truncation=True, padding="max_length"
    )
    labels = tokenizer(
        text_target=examples[HEADLINE_COL],
        max_length=MAX_TARGET_LEN,
        truncation=True,
        padding="max_length",
    )
    labels["input_ids"] = [
        [(tok if tok != tokenizer.pad_token_id else -100) for tok in label]
        for label in labels["input_ids"]
    ]
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = train_data.map(preprocess, batched=True, remove_columns=train_data.column_names)
tokenized_eval = eval_data.map(preprocess, batched=True, remove_columns=eval_data.column_names)

## 6. Configure LoRA

Only a small number of parameters get trained — this is what makes fine-tuning
feasible on a free GPU instead of needing to update the whole model.

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q", "v"],  # attention query/value projections in T5
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 688,128 || all params: 77,649,280 || trainable%: 0.8862


## 7. Metrics (ROUGE — standard for headline/summary quality)

In [ ]:
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    # Ensure predictions are a numpy array and handle non-finite values
    predictions = np.array(predictions)
    predictions[~np.isfinite(predictions)] = tokenizer.pad_token_id

    # Convert to a signed 32-bit integer type, which is often expected by C libraries
    predictions = predictions.astype(np.int32)

    # Clamp predictions to be within valid token ID range
    predictions = np.clip(predictions, 0, tokenizer.vocab_size - 1)

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    return {k: round(v, 4) for k, v in result.items()}

## 8. Training arguments & Trainer

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./headline-lora-checkpoints",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=1e-3,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,
    fp16=False,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    report_to="none",
)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

## 9. Train

This is the part that actually uses the GPU. If your session disconnects,
you'll need to re-run cells 1-8 then this cell again (Colab free tier has no
persistence between disconnects unless you save checkpoints to Drive — see
the optional cell below).

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,3.454297,3.082882,0.310600,0.113200,0.285000,0.285200
2,3.298207,3.063059,0.324100,0.119000,0.294800,0.295400
3,3.282797,3.056768,0.323500,0.121800,0.298600,0.298700
4,3.275639,3.051795,0.319800,0.118000,0.297100,0.298200
5,3.173016,3.048856,0.318500,0.116100,0.294800,0.295600


TrainOutput(global_step=940, training_loss=3.3098009150078957, metrics={'train_runtime': 253.7069, 'train_samples_per_second': 59.123, 'train_steps_per_second': 3.705, 'total_flos': 705016627200000.0, 'train_loss': 3.3098009150078957, 'epoch': 5.0})

## 10. Save the LoRA adapter

In [ ]:
SAVE_PATH = "./headline-lora-adapter"
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print("Saved to", SAVE_PATH)

Saved to ./headline-lora-adapter


## 10b. (Optional) Save to Google Drive so it survives session disconnects

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copytree(SAVE_PATH, "/content/drive/MyDrive/headline-lora-adapter2", dirs_exist_ok=True)
print("Copied to Drive.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copied to Drive.


## 11. Try it out — compare fine-tuned model vs base pretrained model

This is the comparison your project writeup should include: does fine-tuning
actually improve headline quality over just using the pretrained model as-is?

In [ ]:
def generate_headline(text, model_to_use):
    inputs = tokenizer(PREFIX + text, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LEN).to(DEVICE)
    output_ids = model_to_use.generate(**inputs, max_length=MAX_TARGET_LEN, num_beams=4)
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

test_article = (
    "nepal's stock market fell by 3.2 percent on monday after the central bank "
    "announced tighter lending rules for margin trading."
)

print("Fine-tuned model output:", generate_headline(test_article, model))

Fine-tuned model output: nepal stock market falls by 3.2 percent


In [ ]:
# Load the ORIGINAL pretrained model fresh (no LoRA) for comparison
base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)
print("Base pretrained model output:", generate_headline(test_article, base_model))

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Base pretrained model output: nepal stock market falls by 3.2 percent


## 12. (Optional) Evaluate on a larger held-out set for your writeup

Run `trainer.evaluate()` to get final ROUGE scores you can quote directly
in your project report.

In [ ]:
final_metrics = trainer.evaluate()
print(final_metrics)

Training Loss,Validation Loss,Epoch,Rouge1,Rouge2,Rougel,Rougelsum
3.173016,3.056768,5,0.323500,0.121800,0.298600,0.298700


{'eval_loss': 3.0567679405212402, 'eval_rouge1': 0.3235, 'eval_rouge2': 0.1218, 'eval_rougeL': 0.2986, 'eval_rougeLsum': 0.2987}
